# 01 · Coleta de manchetes via RSS

Coleta manchetes dos feeds RSS e filtra as que citam empresas da carteira (`carteira.csv`).

Guardamos só metadados públicos do RSS: manchete, fonte, data e link. Nunca o texto completo.

**Como rodar:** abra no VS Code ou no Jupyter e execute tudo. Não precisa de GPU.
O RSS mostra só as notícias mais recentes, então rode o notebook uma vez por dia: cada execução soma as novas manchetes a `data/raw/manchetes.csv`.

## Configuração

In [ ]:
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote_plus

import feedparser
import pandas as pd

FEEDS = {
    "InfoMoney": "https://www.infomoney.com.br/feed/",
    "Money Times": "https://www.moneytimes.com.br/feed/",
    "Exame": "https://exame.com/feed/",
    "g1 Economia": "https://g1.globo.com/rss/g1/economia/",
}

# Os feeds acima trazem ~150 manchetes por coleta, a maioria sem empresa da carteira.
# A busca do Google News por empresa traz até 100 manchetes de vários veículos cada.
USAR_GOOGLE_NEWS = True

USER_AGENT = "Mozilla/5.0 (portfolio-radar; projeto academico)"

PASTA = Path("../data/raw")  # dados brutos ficam fora do git
PASTA.mkdir(parents=True, exist_ok=True)
ARQ_MANCHETES = PASTA / "manchetes.csv"
ARQ_FILTRADAS = PASTA / "manchetes_filtradas.csv"

## Carteira

Lida de `data/carteira.csv`.

In [ ]:
carteira = pd.read_csv("../data/carteira.csv")
carteira["apelidos_ambiguos"] = carteira["apelidos_ambiguos"].fillna("")
carteira

## Coleta

In [ ]:
def para_iso(data):
    return datetime(*data[:6], tzinfo=timezone.utc).isoformat() if data else None


def ler_feed(url, fonte, rotulo=None):
    d = feedparser.parse(url, agent=USER_AGENT)
    linhas = []
    for e in d.entries:
        manchete = e.get("title", "").strip()
        veiculo = fonte
        # No Google News a manchete vem como "Título - Veículo"
        if fonte == "Google News" and "source" in e:
            veiculo = e.source.get("title", "")
            manchete = manchete.removesuffix(f" - {veiculo}").strip()
            veiculo = f"Google News / {veiculo}"
        linhas.append({
            "manchete": manchete,
            "fonte": veiculo,
            "data": para_iso(e.get("published_parsed") or e.get("updated_parsed")),
            "link": e.get("link"),
            "coletado_em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        })
    print(f"{rotulo or fonte:<40} {len(linhas):>4} manchetes")
    return linhas


def url_google_news(empresa, ticker):
    q = quote_plus(f'"{empresa}" OR {ticker}')
    return f"https://news.google.com/rss/search?q={q}&hl=pt-BR&gl=BR&ceid=BR:pt-419"


linhas = []
for fonte, url in FEEDS.items():
    linhas += ler_feed(url, fonte)

if USAR_GOOGLE_NEWS:
    for _, emp in carteira.iterrows():
        rotulo = f"Google News: {emp['empresa']}"
        linhas += ler_feed(url_google_news(emp["empresa"], emp["ticker"]), "Google News", rotulo)
        time.sleep(1)

novas = pd.DataFrame(linhas)
novas = novas[novas["manchete"].ne("") & novas["link"].notna()]
print(f"\nColetadas agora: {len(novas)}")

## Junta com as coletas anteriores e remove duplicatas

A mesma notícia aparece em mais de um feed: removemos por link e por manchete igual (ignorando maiúsculas e espaços).

In [ ]:
if ARQ_MANCHETES.exists():
    anteriores = pd.read_csv(ARQ_MANCHETES)
    todas = pd.concat([anteriores, novas], ignore_index=True)
else:
    todas = novas

chave = todas["manchete"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
todas = todas[~todas["link"].duplicated() & ~chave.duplicated()].reset_index(drop=True)
todas.to_csv(ARQ_MANCHETES, index=False)
print(f"Total acumulado: {len(todas)} manchetes em {ARQ_MANCHETES}")

## Filtro pelas empresas da carteira

- Tickers e nomes comuns: busca sem diferenciar maiúsculas.
- Apelidos ambíguos (coluna `apelidos_ambiguos`, ex.: "Vale", "Vivo", "Localiza"): exige a grafia exata com maiúscula. Se a manchete só casa por um desses nomes, sem o ticker ou outro apelido, a linha sai com `revisar = True` para conferência manual.
- Os limites de palavra evitam casar "Itaú" dentro de "Itaúsa".

In [ ]:
def compila(apelido, ambiguo):
    flags = 0 if ambiguo else re.IGNORECASE
    return re.compile(rf"(?<!\w){re.escape(apelido)}(?!\w)", flags)


padroes = []  # (ticker, regex, apelido_ambiguo)
for _, emp in carteira.iterrows():
    ambiguos = set(emp["apelidos_ambiguos"].split("|")) - {""}
    for apelido in emp["apelidos"].split("|"):
        amb = apelido in ambiguos
        padroes.append((emp["ticker"], compila(apelido, amb), amb))


def empresas_citadas(manchete):
    seguras, ambiguas = set(), set()
    for ticker, rx, amb in padroes:
        if rx.search(manchete):
            (ambiguas if amb else seguras).add(ticker)
    return sorted(seguras | ambiguas), bool(ambiguas - seguras)


res = todas["manchete"].apply(empresas_citadas)
todas["empresas"] = res.str[0].str.join("|")
todas["revisar"] = res.str[1]

filtradas = todas[todas["empresas"].ne("")].reset_index(drop=True)
filtradas.to_csv(ARQ_FILTRADAS, index=False)
print(f"{len(filtradas)} manchetes citam a carteira ({filtradas['revisar'].sum()} para revisar)")
print(f"Citam mais de uma empresa: {filtradas['empresas'].str.contains('|', regex=False).sum()}")
print(f"Salvo em {ARQ_FILTRADAS}")

## Resumo

In [ ]:
por_empresa = filtradas["empresas"].str.split("|").explode().value_counts()
por_empresa.rename(index=carteira.set_index("ticker")["empresa"]).to_frame("manchetes")

In [ ]:
filtradas["fonte"].str.replace(r"^Google News / .*", "Google News", regex=True).value_counts().to_frame("manchetes")

In [ ]:
filtradas.sample(min(15, len(filtradas)), random_state=0)[["manchete", "fonte", "empresas", "revisar"]]